# SQL Intermediate

**Estimated time:** ~12 hours total (about 5 hours reading and running this guide + ~7 hours on
`sql-intermediate-exercises.ipynb`).

The basics level got you one table at a time and a summary at the end. This level is about the questions that
do not fit that shape: *which product was our best seller in each category, each month?* *How much did revenue
change month over month?* *Which customers came back?*

Three tools do almost all of that work — subqueries, common table expressions, and window functions — and the
first thing this guide does is show you the bug that quietly wrecks the answers before you learn them.

## Who This Is For

You finished `sql-basics`, or you already write `SELECT`, `WHERE`, `GROUP BY` and joins without looking them
up. You have probably written a query that returned a number you could not fully justify. That number is
section 3.

## What You Will Learn

- Every join type, including the self-join, and what `CROSS JOIN` is actually for
- **Row fan-out** — why joining before aggregating doubles your revenue, and the two ways to fix it
- Anti-joins and semi-joins, and the `NOT IN` bug that returns zero rows and no error
- Subqueries in every position they can appear, including correlated ones
- `UNION`, `INTERSECT`, `EXCEPT`
- CTEs (`WITH`), and how they turn a frightening query into a readable list of steps
- Window functions: `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, running totals, moving averages, `NTILE`
- Conditional aggregation — building a cross-tab without a `PIVOT` keyword
- Cohorts, retention, and the date work that goes with them
- Views, bulk loads, upserts and transactions
- How to lay a long query out so the next person can read it

## How to Use This Guide

Run every cell with **Shift + Enter**, in order. The setup cell builds the same shop database you used in the
basics level, in memory, from `../assets/sql/`.

Predict before you run. From here on the queries return things like "one row per product per month", and the
habit of stating the grain of the result out loud — *what does one row of this mean?* — is most of what
separates a correct query from a plausible one.

When a query has more than two steps, write it as a CTE chain and run each step on its own first. That is not
a beginner's crutch; it is how the queries in this guide were built.

**Practice:** `sql-intermediate-exercises.ipynb` follows this guide section by section.

## 1. Setup — The Same Shop, One Level Up

The database is unchanged from `sql-basics`: seven tables, 300 orders, 673 order lines, and a set of
deliberate imperfections that this level finally gives you the tools to deal with.

Three of those imperfections drive most of what follows. Nine customers have never ordered. Fifty-two orders
were never paid for. Two people signed up twice with the same email address. Every one of them is the kind of
thing that turns an honest-looking `JOIN` into a wrong number.

Run the cell to build it.

In [2]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")

categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows


**Step by step:**

1. `sqlite3.connect(":memory:")` builds the database in RAM. It is gone when the kernel stops and rebuilt the
   moment you re-run this cell, so nothing you do later in the notebook can leave you stuck.
2. `executescript` runs `schema.sql`, creating the seven tables with their types, primary keys and foreign keys.
3. `PRAGMA foreign_keys = ON` comes after that, so the drops at the top of the script are not blocked by the
   references they are about to remove.
4. The loop loads each CSV in dependency order — parents before children — because with foreign keys on, an
   order cannot be inserted before the customer it points at.
5. `q(sql)` returns a `SELECT` as a DataFrame, `run(sql)` executes statements that change things. The row
   counts printed at the end are your check that everything loaded.

## 2. Every Join Type — And What Each One Is Actually For

Five joins, and you will use three of them.

| Join | Keeps | Use it when |
| --- | --- | --- |
| `INNER JOIN` | rows matched on both sides | the match is guaranteed, or you want only the matched ones |
| `LEFT JOIN` | all of the left, matched or not | the right side is optional — payments, reps, anything nullable |
| `RIGHT JOIN` | all of the right | never, in practice — swap the tables and use `LEFT` |
| `FULL OUTER JOIN` | everything from both sides | reconciling two lists that each have rows the other lacks |
| `CROSS JOIN` | every combination | building a complete grid of dates × categories to report against |

A **self-join** is not a sixth type. It is any of the above with the same table on both sides, which is how you
walk a hierarchy: `employees` joined to `employees` turns `manager_id` into a manager's name.

`CROSS JOIN` deserves a word, because it looks like a mistake and is not. When you need a report with a row for
every month and every channel — including the empty combinations — you cross join the months against the
channels and `LEFT JOIN` the data onto that grid. Without it, months where a channel sold nothing simply do not
appear, and a chart with missing bars looks like a chart with zeros.

`RIGHT` and `FULL OUTER` joins only arrived in SQLite 3.39. If yours is older they will not parse, which is one
more reason `LEFT JOIN` is the one to reach for.

In [ ]:
inner_n = q("SELECT COUNT(*) AS n FROM customers c JOIN customers o ON o.customer_id = c.customer_id")
print("customers                   ", q("SELECT COUNT(*) AS n FROM customers")["n"][0])
print("INNER JOIN orders           ", q("""
    SELECT COUNT(DISTINCT c.customer_id) AS n
    FROM customers c JOIN orders o ON o.customer_id = c.customer_id""")["n"][0], "customers survive")
print("LEFT  JOIN orders           ", q("""
    SELECT COUNT(DISTINCT c.customer_id) AS n
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id""")["n"][0], "customers survive")
print("CROSS JOIN categories x days", q("""
    SELECT COUNT(*) AS n
    FROM categories CROSS JOIN (SELECT DISTINCT channel FROM orders)""")["n"][0], "combinations")

q("""
SELECT e.employee_id,
       e.name  AS employee,
       e.role,
       m.name  AS manager
FROM employees e
LEFT JOIN employees m ON m.employee_id = e.manager_id
ORDER BY e.employee_id
LIMIT 8
""")

**Step by step:**

1. The `INNER JOIN` count is 51, the `LEFT JOIN` count is 60. That nine-row gap is the nine customers who never
   ordered, and an inner join loses them without saying so.
2. `COUNT(DISTINCT c.customer_id)` rather than `COUNT(*)` because the join produces one row per order — the
   subject of the next section.
3. The `CROSS JOIN` pairs 8 categories with 4 channels and returns 32 rows. No `ON` clause: there is no
   condition, every left row meets every right row.
4. The last query joins `employees` to itself. `e` is the employee, `m` is their manager, and
   `ON m.employee_id = e.manager_id` is the whole trick — the same table read through two different aliases.
5. It has to be a `LEFT JOIN`: the founder's `manager_id` is `NULL`, and an inner join would drop the one row at
   the top of the tree. Hierarchies always have that row.

## 3. Row Fan-Out — Why Your SUM Doubled

Here is the bug that produces confident, wrong numbers in production dashboards.

An order has one payment. An order has several items. Join `payments` to `order_items` and each payment row is
repeated once per item — a three-item order makes the payment appear three times. `SUM(amount)` now counts the
same money three times over.

Nothing warns you. The query is valid, the rows look right, and the total is wrong by a factor that changes
with the data.

Two fixes, and it is worth knowing both:

1. **Aggregate first, join second.** Collapse each table to one row per order in its own subquery or CTE, then
   join the two summaries. This is the one to reach for.
2. **Aggregate the fanned column separately** with `COUNT(DISTINCT ...)` or `SUM(DISTINCT ...)`. `COUNT(DISTINCT
   order_id)` is genuinely useful; `SUM(DISTINCT amount)` is a trap, because two orders that legitimately paid
   the same amount collapse into one.

The habit that prevents all of it: before you write `SUM`, say what one row of your `FROM` clause means. If it
means "one item line", you cannot sum an order-level column over it.

In [ ]:
truth = q("SELECT ROUND(SUM(amount), 2) AS paid FROM payments")["paid"][0]

fanned = q("""
SELECT ROUND(SUM(p.amount), 2) AS paid
FROM payments p
JOIN order_items i ON i.order_id = p.order_id
""")["paid"][0]

fixed = q("""
WITH item_count AS (
    SELECT order_id, COUNT(*) AS lines
    FROM order_items
    GROUP BY order_id
)
SELECT ROUND(SUM(p.amount), 2) AS paid
FROM payments p
JOIN item_count c ON c.order_id = p.order_id
""")["paid"][0]

print(f"payments alone            {truth:>14,.2f}")
print(f"joined to order_items     {fanned:>14,.2f}   <- {fanned / truth:.2f}x too big")
print(f"aggregate first, then join{fixed:>14,.2f}")

q("""
SELECT p.order_id,
       COUNT(*)                  AS rows_after_join,
       ROUND(MIN(p.amount), 2)   AS actual_payment,
       ROUND(SUM(p.amount), 2)   AS what_sum_would_give
FROM payments p
JOIN order_items i ON i.order_id = p.order_id
GROUP BY p.order_id
ORDER BY rows_after_join DESC, p.order_id
LIMIT 5
""")

**Step by step:**

1. `truth` sums `payments` on its own. One row per payment, so the total is the money that came in. This is the
   number to match.
2. `fanned` joins to `order_items` first. The total is more than twice as large, because the average order has
   more than two lines and each one duplicates its payment row.
3. `fixed` collapses `order_items` to one row per order **before** joining, so the join is one-to-one again and
   the sum is correct. The `WITH ... AS (...)` block is a CTE — section 11 covers them properly.
4. The last query shows the mechanism per order: `rows_after_join` is how many times the payment was
   duplicated, `actual_payment` is what it really was, `what_sum_would_give` is that number multiplied.
5. Note the fix does not change the join — it changes the **grain** of what is being joined. Grain is the word
   for "what one row means", and keeping track of it is the whole skill.

## 4. Anti-Joins and Semi-Joins — Rows With No Partner

Two shapes you will write constantly:

- A **semi-join** asks "does a match exist?" and returns the left rows that have one — *without* duplicating
  them. `EXISTS` is the natural spelling.
- An **anti-join** asks the opposite: left rows with **no** match. Customers who never ordered, products never
  sold, orders never paid.

Three ways to write an anti-join, and they are not equally good:

```sql
-- 1. LEFT JOIN and test the missing side
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL

-- 2. NOT EXISTS
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)

-- 3. NOT IN
WHERE c.customer_id NOT IN (SELECT customer_id FROM orders)
```

The first two are always correct. The third is correct only when the subquery can never return `NULL`, and when
it does return one the answer is silently empty. That is the next section, and it is worth the separate section.

Use `NOT EXISTS`. It says what you mean, it is safe against `NULL`s, and it stops as soon as it finds one match
rather than building a list.

In [ ]:
by_left_join = q("""
SELECT COUNT(*) AS n
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
""")["n"][0]

by_not_exists = q("""
SELECT COUNT(*) AS n
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
""")["n"][0]

by_not_in = q("""
SELECT COUNT(*) AS n
FROM customers c
WHERE c.customer_id NOT IN (SELECT customer_id FROM orders)
""")["n"][0]

print("LEFT JOIN ... IS NULL:", by_left_join)
print("NOT EXISTS           :", by_not_exists)
print("NOT IN               :", by_not_in, "  (safe here -- orders.customer_id is NOT NULL)")

q("""
SELECT p.product_id, p.name, p.price
FROM products p
WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.product_id = p.product_id)
ORDER BY p.product_id
""")

**Step by step:**

1. All three counts agree at 9. `orders.customer_id` is declared `NOT NULL`, so `NOT IN` is safe here — that is
   the condition, and it is a condition you have to check every time.
2. `SELECT 1` inside `EXISTS` is the convention. `EXISTS` only cares whether a row came back, never what was in
   it, so there is no reason to select a column.
3. The correlation is `WHERE o.customer_id = c.customer_id` — the inner query refers to `c` from the outer one.
   That is what makes it run per outer row, and section 8 goes into it.
4. The last query is the anti-join in its most useful form: two products that have never been ordered. An
   `INNER JOIN` would have hidden them, which is exactly why nobody notices dead stock.
5. Semi-join version: drop the `NOT` and you have "products that have sold at least once", still one row per
   product no matter how many times each one sold. A `JOIN` would have given you one row per sale.

## 5. NOT IN and NULL — The Bug That Returns Nothing

`x IN (1, 2, NULL)` is true when `x` is 1 or 2, and **unknown** otherwise — because SQL cannot rule out that
the `NULL` is secretly equal to `x`.

Now negate it. `x NOT IN (1, 2, NULL)` is false when `x` is 1 or 2, and **unknown** for everything else. Never
true. And `WHERE` keeps only what is true.

So: **if the subquery inside a `NOT IN` returns even one `NULL`, the whole query returns zero rows.** No error,
no warning, no clue. Somebody adds a nullable column to a table, and a report that has been right for two years
starts returning an empty result.

Our `orders.employee_id` is `NULL` for every web and app order, which makes this easy to demonstrate.

The fix is always the same: use `NOT EXISTS`, or add `WHERE column IS NOT NULL` inside the subquery. `NOT
EXISTS` is the better habit because it needs no thought.

In [ ]:
broken = q("""
SELECT COUNT(*) AS n
FROM employees e
WHERE e.employee_id NOT IN (SELECT employee_id FROM orders)
""")["n"][0]

patched = q("""
SELECT COUNT(*) AS n
FROM employees e
WHERE e.employee_id NOT IN (SELECT employee_id FROM orders WHERE employee_id IS NOT NULL)
""")["n"][0]

correct = q("""
SELECT COUNT(*) AS n
FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.employee_id = e.employee_id)
""")["n"][0]

print("NOT IN, subquery has NULLs :", broken, " <- wrong, and it looks like a real answer")
print("NOT IN with IS NOT NULL    :", patched)
print("NOT EXISTS                 :", correct)

q("""
SELECT e.employee_id, e.name, e.role
FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.employee_id = e.employee_id)
ORDER BY e.employee_id
""")

**Step by step:**

1. The first query returns 0. There are certainly employees who never took an order — the founder and the
   support team — so 0 is wrong, and it arrives without a single complaint from the database.
2. The cause is the `NULL`s in `orders.employee_id`. One `NULL` in the list is enough to make every `NOT IN`
   test unknown, and unknown is not true.
3. Adding `WHERE employee_id IS NOT NULL` inside the subquery fixes it — 7 employees, the right answer.
4. `NOT EXISTS` gets the same 7 without needing you to have thought about it at all. It compares row by row, and
   a `NULL` simply fails to match rather than poisoning the whole test.
5. The listing at the end shows who they are: the founder, three managers and three support agents. Sales reps
   are missing from it because they all took at least one order, which is the sanity check that says the answer
   is right.

## 6. Subqueries in WHERE — Scalar and List

A subquery is a `SELECT` inside another statement. In a `WHERE` clause it comes in two shapes:

- **Scalar** — returns exactly one row and one column, and is used as a value:
  `WHERE price > (SELECT AVG(price) FROM products)`. If it ever returns more than one row you get an error, and
  if it returns none you get `NULL`, which quietly matches nothing.
- **List** — returns one column and any number of rows, used with `IN`:
  `WHERE category_id IN (SELECT category_id FROM categories WHERE name LIKE 'A%')`.

Both are evaluated by the database when it needs them, which for an uncorrelated subquery like these is once
for the whole query — not once per row.

In [ ]:
print(q("SELECT ROUND(AVG(price), 2) AS avg_price FROM products").to_string(index=False))

q("""
SELECT p.name,
       p.price,
       c.name AS category
FROM products p
JOIN categories c ON c.category_id = p.category_id
WHERE p.price > (SELECT AVG(price) FROM products)
  AND p.category_id IN (SELECT category_id FROM categories
                        WHERE name IN ('Laptops', 'Monitors', 'Cameras'))
ORDER BY p.price DESC, p.name
""")

**Step by step:**

1. `(SELECT AVG(price) FROM products)` is scalar — one number, about 26,500. The outer query compares every
   product's price against it.
2. The brackets are required. Without them SQL cannot tell where the inner query ends.
3. `IN (SELECT category_id FROM categories WHERE name IN (...))` is the list form. It could have been written as
   a join, and often should be, but as a filter it reads well and does not risk fan-out.
4. Note the two different `IN`s: one over a subquery, one over a literal list. Same operator, same meaning.
5. The scalar subquery runs once, not once per product. You can check that intuition in the advanced level with
   `EXPLAIN QUERY PLAN`, which will show it as a one-off scan.

## 7. Derived Tables — A Subquery in the FROM

A subquery in the `FROM` clause is a **derived table**: a result treated as if it were a table. It needs a name.

This is how you aggregate an aggregate. "The average order value" is really "the average of the per-order
totals", and per-order totals are themselves a `GROUP BY`. You cannot nest aggregates directly —
`AVG(SUM(x))` is not a thing — so you compute the inner summary first, then aggregate that.

```sql
SELECT channel, AVG(order_total)
FROM ( SELECT o.order_id, o.channel, SUM(...) AS order_total
       FROM orders o JOIN order_items i ...
       GROUP BY o.order_id, o.channel ) t
GROUP BY channel
```

The inner query's grain is one row per order. The outer query's grain is one row per channel. Stating both out
loud before you write either one is the fastest way to get these right.

Derived tables nest, and once you are two deep they become unreadable. That is what CTEs in section 11 are for.

In [ ]:
q("""
SELECT t.channel,
       COUNT(*)                     AS orders,
       ROUND(AVG(t.order_total), 2) AS avg_order_value,
       ROUND(MAX(t.order_total), 2) AS biggest_order
FROM (
    SELECT o.order_id,
           o.channel,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS order_total
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.order_id, o.channel
) t
GROUP BY t.channel
ORDER BY avg_order_value DESC
""")

**Step by step:**

1. The inner query joins orders to their items and groups by `order_id`, so one row means **one order** and
   `order_total` is that order's value. `channel` is in the `GROUP BY` because it has to travel outwards.
2. `) t` names the derived table. Most databases insist on a name even when you never use it; give it a real one
   rather than `t` in queries you keep.
3. The outer query groups those order rows by channel. `COUNT(*)` here counts **orders**, not item lines,
   because the inner query already collapsed the lines.
4. `AVG(t.order_total)` is the average order value — the thing you could not have written as `AVG(SUM(...))`.
5. Try removing the inner query and computing `AVG(i.quantity * i.unit_price)` directly. You get the average
   *line* value, which is a different and much smaller number. Same tables, different grain, different question.

## 8. Correlated Subqueries — One Answer per Row

An ordinary subquery is independent: it runs once. A **correlated** subquery mentions a column from the outer
query, so it has to be evaluated again for every outer row.

```sql
SELECT p.name, p.price,
       (SELECT AVG(p2.price) FROM products p2 WHERE p2.category_id = p.category_id) AS category_avg
FROM products p
```

That inner query cannot run on its own — it needs a `p` to exist. Conceptually the database loops over the
outer rows and runs it each time, which is why correlated subqueries are the slow option when the outer query
is large.

They are also, usually, a window function in disguise. The query above is
`AVG(price) OVER (PARTITION BY category_id)`, which does the same job in a single pass. Section 12 makes that
swap. Learn the correlated form anyway — it appears in a great deal of existing code, and `EXISTS` is a
correlated subquery you will keep using forever.

In [ ]:
q("""
SELECT p.name,
       p.category_id,
       p.price,
       (SELECT ROUND(AVG(p2.price), 2)
        FROM products p2
        WHERE p2.category_id = p.category_id) AS category_avg,
       (SELECT COUNT(*)
        FROM order_items i
        WHERE i.product_id = p.product_id)    AS times_ordered
FROM products p
WHERE p.price > (SELECT AVG(p2.price)
                 FROM products p2
                 WHERE p2.category_id = p.category_id)
ORDER BY p.category_id, p.price DESC
LIMIT 10
""")

**Step by step:**

1. The first subquery computes the average price **of the row's own category**, because of
   `WHERE p2.category_id = p.category_id`. Different row, different category, different answer.
2. `p2` is a second alias for the same table. Without it there would be no way to say "the other products"
   as distinct from "this product".
3. The second subquery counts order lines for this product. A correlated subquery in the `SELECT` list like
   this returns one value per row and is the classic "lookup" shape.
4. The `WHERE` uses the same correlated average again, keeping only products priced above their own category's
   average. Written out twice here for clarity; a CTE or a window function would compute it once.
5. On 40 products the repetition is free. On 40 million rows it is three full passes, and the advanced level
   shows you how to see that in the query plan.

## 9. EXISTS — The Question That Stops at the First Match

`EXISTS (subquery)` is true if the subquery returns at least one row. `NOT EXISTS` is true if it returns none.
What the subquery selects is irrelevant, which is why everybody writes `SELECT 1`.

Two reasons to prefer it over the alternatives:

- **It cannot fan out.** `WHERE EXISTS (...)` returns each outer row at most once, however many matches there
  are. Rewriting the same question as a `JOIN` gives you a row per match and then you need `DISTINCT` to
  clean up after yourself.
- **It short-circuits.** The database stops looking as soon as one match is found.

The shape to remember: *"customers who have ever bought a laptop"* is `EXISTS`, not a join. *"every laptop
purchase by every customer"* is a join. Ask whether you want the customers or the purchases.

In [ ]:
with_exists = q("""
SELECT c.customer_id, c.name, c.city
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    JOIN products p    ON p.product_id = i.product_id
    WHERE o.customer_id = c.customer_id
      AND p.category_id = 1
)
ORDER BY c.customer_id
""")

with_join = q("""
SELECT c.customer_id, c.name, c.city
FROM customers c
JOIN orders o      ON o.customer_id = c.customer_id
JOIN order_items i ON i.order_id = o.order_id
JOIN products p    ON p.product_id = i.product_id
WHERE p.category_id = 1
ORDER BY c.customer_id
""")

print("EXISTS returns", len(with_exists), "rows -- one per customer")
print("JOIN   returns", len(with_join), "rows -- one per laptop purchased")
print("after DISTINCT:", with_join.drop_duplicates().shape[0])

with_exists.head(8)

**Step by step:**

1. Both queries answer "which customers have bought a laptop". `EXISTS` returns 30 rows; the join returns 68,
   because a customer who bought several laptops appears once per purchase.
2. `drop_duplicates()` on the join result gets back to 30 — the same answer, reached by making a mess and then
   tidying it. `SELECT DISTINCT` in SQL is the same tidying.
3. The correlation `WHERE o.customer_id = c.customer_id` is what ties the subquery to the row being tested.
   Leave it out and the subquery is true for everybody, so every customer comes back.
4. Everything else inside the `EXISTS` is an ordinary three-table join. Subqueries can be as complicated as you
   like; only the correlation makes them special.
5. When you want facts *about* the matches — how many laptops, how much they spent — you need the join and a
   `GROUP BY`. `EXISTS` answers yes or no, and nothing else.

## 10. UNION, INTERSECT and EXCEPT — Stacking Results

These combine two result sets **vertically**, where a join combines them horizontally. Both sides must have the
same number of columns, in the same order, with compatible types. The column names come from the first query.

- `UNION ALL` stacks everything. Fast, keeps duplicates.
- `UNION` stacks and then removes duplicate rows. That deduplication costs a sort, so do not pay for it out of
  habit — if the two sides cannot overlap, use `UNION ALL`.
- `INTERSECT` keeps rows appearing in both.
- `EXCEPT` keeps rows in the first that are not in the second. (`MINUS` in Oracle.)

The everyday use of `UNION ALL` is stacking a summary onto its own detail — a totals row under a breakdown —
and combining tables that hold the same shape of thing for different periods.

In [ ]:
print("cities we sell to but do not staff:")
print(q("""
SELECT city FROM customers WHERE city IS NOT NULL
EXCEPT
SELECT city FROM employees
ORDER BY city
""").to_string(index=False))

print("\ncities that are both:")
print(q("""
SELECT city FROM customers WHERE city IS NOT NULL
INTERSECT
SELECT city FROM employees
ORDER BY city
""").to_string(index=False))

q("""
SELECT channel, COUNT(*) AS orders
FROM orders
GROUP BY channel

UNION ALL

SELECT 'TOTAL', COUNT(*)
FROM orders

ORDER BY orders DESC
""")

**Step by step:**

1. `EXCEPT` gives the customer cities with no employee in them. Both sides return one column called `city`, so
   the shapes line up.
2. `WHERE city IS NOT NULL` on the first side keeps the `NULL` city out of a list of place names. `EXCEPT` and
   `INTERSECT` treat two `NULL`s as the same value, which is different from how `=` treats them — another
   corner of `NULL` behaviour that is simply worth knowing.
3. `INTERSECT` gives the cities in both lists.
4. The last query stacks a grand total under the per-channel breakdown, with the literal `'TOTAL'` standing in
   for the channel name. `UNION ALL` rather than `UNION` because the two sides cannot possibly duplicate.
5. The `ORDER BY` at the end sorts the combined result, not either half. It belongs to the whole statement, and
   putting one on the first `SELECT` is a syntax error in most databases.

## 11. CTEs — Naming the Steps

A **common table expression** is a named subquery written before the query that uses it:

```sql
WITH order_totals AS (
    SELECT ... FROM ... GROUP BY ...
),
monthly AS (
    SELECT ... FROM order_totals GROUP BY ...
)
SELECT * FROM monthly WHERE ...
```

Anything you can write as a nested subquery you can write as a CTE, and you should. The difference is entirely
readability, and readability is the difference between a query somebody can fix and one they rewrite from
scratch.

Three real advantages:

- Each step gets a **name**, so the query reads as a list of steps instead of an onion.
- A CTE can be referenced **more than once** in the same query. A derived table cannot.
- You can develop them one at a time: write the first CTE, `SELECT * FROM` it, check the grain, then add the
  next.

CTEs can also be recursive, which is how you walk a hierarchy or generate a series of dates. That is a topic in
`sql-advanced`.

In [ ]:
q("""
WITH order_totals AS (
    SELECT o.order_id,
           o.customer_id,
           o.order_date,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS order_total
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.order_id, o.customer_id, o.order_date
),
per_customer AS (
    SELECT customer_id,
           COUNT(*)         AS orders,
           SUM(order_total) AS lifetime_value,
           MAX(order_date)  AS last_order
    FROM order_totals
    GROUP BY customer_id
)
SELECT c.name,
       COALESCE(c.city, 'unknown')  AS city,
       pc.orders,
       ROUND(pc.lifetime_value, 2)  AS lifetime_value,
       pc.last_order
FROM per_customer pc
JOIN customers c ON c.customer_id = pc.customer_id
ORDER BY pc.lifetime_value DESC
LIMIT 8
""")

**Step by step:**

1. `order_totals` is step one: one row per order, with its value. Run this CTE on its own — replace everything
   after it with `SELECT * FROM order_totals LIMIT 5` — and you can see its grain directly.
2. `per_customer` is step two, and it reads from `order_totals` as if it were a table. Each CTE after the first
   is separated by a comma; there is no second `WITH`.
3. Because `order_totals` already collapsed the item lines, `COUNT(*)` in `per_customer` counts orders. The
   fan-out from section 3 cannot happen — the steps fixed the grain before the counting started.
4. The final `SELECT` joins the summary back to `customers` for the names. Joining a small summary to a lookup
   table at the end is the standard shape.
5. The same query as nested subqueries would be three levels of brackets read inside out. This one reads top to
   bottom, which is the entire argument for CTEs.

## 12. Window Functions — Aggregates That Keep Your Rows

`GROUP BY` collapses rows. A **window function** computes across a set of rows and gives the answer back
**on every row**, changing nothing about the shape of the result.

```sql
AVG(price) OVER (PARTITION BY category_id)
```

Read `OVER` as "looking at". `PARTITION BY` says which rows to look at — the same idea as `GROUP BY`, except
nothing is collapsed. Leave the `OVER ()` empty and the window is the entire result set.

That is what makes "this row compared to its group" a single query: the row's own value and its group's
average sit side by side, with no subquery and no self-join.

Anything you can aggregate you can window — `SUM`, `AVG`, `COUNT`, `MIN`, `MAX` — plus a family that only
exists as window functions: `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, `NTILE`, the next four sections.

One rule that catches everybody: **window functions run after `WHERE`, `GROUP BY` and `HAVING`, and before
`ORDER BY`.** So you cannot filter on one in the same query's `WHERE` — `WHERE ROW_NUMBER() OVER (...) = 1` is
an error. Wrap the query in a CTE and filter outside. You will do that in every top-N query you ever write.

In [ ]:
q("""
SELECT c.name AS category,
       p.name AS product,
       p.price,
       ROUND(AVG(p.price)  OVER (PARTITION BY p.category_id), 2) AS category_avg,
       ROUND(p.price - AVG(p.price) OVER (PARTITION BY p.category_id), 2) AS vs_avg,
       COUNT(*)            OVER (PARTITION BY p.category_id)      AS in_category,
       ROUND(100.0 * p.price / SUM(p.price) OVER (PARTITION BY p.category_id), 1) AS pct_of_category,
       ROUND(AVG(p.price)  OVER (), 2)                            AS overall_avg
FROM products p
JOIN categories c ON c.category_id = p.category_id
WHERE p.category_id IN (1, 8)
ORDER BY p.category_id, p.price DESC
""")

**Step by step:**

1. Every row of the result is still one product — eight rows for two categories. Nothing was collapsed.
2. `AVG(p.price) OVER (PARTITION BY p.category_id)` is the average within the row's own category. The laptops
   all show one value, the wearables another.
3. `p.price - AVG(...) OVER (...)` mixes a row value and a window value in one expression, which is the thing
   `GROUP BY` cannot do. Doing this with `GROUP BY` needs a second query and a join back.
4. `SUM(p.price) OVER (PARTITION BY ...)` in the denominator gives each product's share of its category — the
   "percent of total" column that every report wants and that is painful without windows.
5. `OVER ()` with an empty bracket windows over everything, so `overall_avg` is the same on all eight rows. The
   partition is optional; the `OVER` is not.

## 13. ROW_NUMBER, RANK and DENSE_RANK — Top N per Group

Three numbering functions, differing only in how they handle ties:

| Function | Ties get | Sequence with a tie for 2nd |
| --- | --- | --- |
| `ROW_NUMBER()` | different numbers, arbitrarily assigned | 1, 2, 3, 4 |
| `RANK()` | the same number, then a gap | 1, 2, 2, 4 |
| `DENSE_RANK()` | the same number, no gap | 1, 2, 2, 3 |

All three need `OVER (ORDER BY ...)`, because numbering means nothing without an order. Add `PARTITION BY` and
the numbering restarts in each group.

**Top-N-per-group** is the pattern this exists for, and it is worth memorising as a shape:

```sql
WITH ranked AS (
    SELECT ..., ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY sort_col DESC) AS rn
    FROM ...
)
SELECT * FROM ranked WHERE rn <= 3
```

The CTE is not decoration. You cannot filter on a window function in the same query that computes it, so the
wrapper is compulsory.

Which function? `ROW_NUMBER` when you need exactly N rows and do not care who wins a tie — deduplication, for
instance. `RANK` when a tie should genuinely share a position. `DENSE_RANK` when you want "the top 3 distinct
values" rather than the top 3 rows.

In [ ]:
q("""
WITH product_sales AS (
    SELECT p.category_id,
           p.name AS product,
           SUM(i.quantity)                                     AS units,
           SUM(i.quantity * i.unit_price * (1 - i.discount))    AS revenue
    FROM order_items i
    JOIN products p ON p.product_id = i.product_id
    JOIN orders o   ON o.order_id   = i.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY p.category_id, p.name
),
ranked AS (
    SELECT ps.*,
           ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY revenue DESC, product) AS rn,
           RANK()       OVER (PARTITION BY category_id ORDER BY units   DESC)          AS units_rank
    FROM product_sales ps
)
SELECT c.name AS category,
       product,
       units,
       ROUND(revenue, 2) AS revenue,
       rn,
       units_rank
FROM ranked r
JOIN categories c ON c.category_id = r.category_id
WHERE rn <= 2
ORDER BY r.category_id, rn
""")

**Step by step:**

1. `product_sales` gets the grain right first: one row per product, with its units and revenue. Cancelled
   orders are excluded here, once, rather than in three places later.
2. `ranked` numbers those rows. `PARTITION BY category_id` restarts the count in each category;
   `ORDER BY revenue DESC, product` decides the order, with the product name as a tiebreaker so the result is
   reproducible.
3. `ps.*` selects everything from the CTE and adds the two new columns beside it — a useful shorthand when the
   inner query already has the columns you want.
4. `WHERE rn <= 2` in the **outer** query keeps the top two per category. This is the reason for the second CTE;
   putting that condition inside `ranked` would be an error.
5. `units_rank` orders by units instead of revenue, so you can see the two disagree — the cheap product that
   sells in volume ranks first on units and nowhere near first on revenue.

## 14. LAG and LEAD — Comparing a Row with Its Neighbour

`LAG(column, n, default)` reaches back `n` rows in the window; `LEAD` reaches forward. Both default to `n = 1`.

This is how month-over-month change is written. Without them you would join a table to itself on
"month minus one", which needs date arithmetic, breaks on gaps, and reads badly.

```sql
LAG(revenue) OVER (ORDER BY month)
```

The first row has nothing behind it, so `LAG` returns `NULL` — and `revenue - NULL` is `NULL`, which is
correct: the change from an unknown previous month is unknown. Supplying a default of `0` instead would claim
the first month grew from zero, which is a different and usually false statement.

Add `PARTITION BY` to compare within a group — each product's own previous month rather than the shop's.

In [ ]:
q("""
WITH monthly AS (
    SELECT strftime('%Y-%m', o.order_date)                        AS month,
           SUM(i.quantity * i.unit_price * (1 - i.discount))      AS revenue
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY month
)
SELECT month,
       ROUND(revenue, 2)                                  AS revenue,
       ROUND(LAG(revenue)  OVER (ORDER BY month), 2)      AS prev_month,
       ROUND(revenue - LAG(revenue) OVER (ORDER BY month), 2) AS change,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1)     AS pct_change,
       ROUND(LEAD(revenue) OVER (ORDER BY month), 2)      AS next_month
FROM monthly
ORDER BY month
LIMIT 8
""")

**Step by step:**

1. The `monthly` CTE reduces everything to one row per month. `LAG` needs a clean, ordered series to walk, so
   getting to that grain first is the whole job.
2. `LAG(revenue) OVER (ORDER BY month)` is the previous month's revenue on the same row. The `ORDER BY` inside
   `OVER` defines "previous"; it is unrelated to the `ORDER BY` at the end of the query.
3. `revenue - LAG(revenue) OVER (...)` is the change. The first row is `NULL`, honestly, because there is no
   month before it.
4. The percentage repeats `LAG(...)` twice, which is ugly. A second CTE that computes `prev_month` once and a
   third that divides by it reads far better, and costs nothing.
5. `LEAD` looks the other way. It is what you want for "how long until the next order" style questions — pair
   it with `PARTITION BY customer_id` and you have the gap between one customer's consecutive orders.

## 15. Frames — Running Totals and Moving Averages

By default, a window function with an `ORDER BY` inside `OVER` looks at every row from the start of the
partition up to **the current row**. That default is what makes a plain `SUM(x) OVER (ORDER BY month)` a
running total.

You can say it explicitly, and for anything else you must:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW   -- running total (the default)
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW           -- 3-month moving average
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- the whole partition
```

The trap is that **the default frame changes when you add `ORDER BY`**. `SUM(x) OVER (PARTITION BY c)` is the
partition total, the same on every row. Add `ORDER BY d` and the same expression silently becomes a running
total. If you want the partition total *and* an ordering, you have to write the frame out.

`ROWS` counts physical rows. `RANGE` counts by value, so tied rows are treated as one unit. `ROWS` is what you
almost always mean; `sql-advanced` covers the difference where it matters.

In [ ]:
q("""
WITH monthly AS (
    SELECT strftime('%Y-%m', o.order_date)                   AS month,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY month
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY month), 2)                          AS running_total,
       ROUND(AVG(revenue) OVER (ORDER BY month
                                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3m,
       ROUND(SUM(revenue) OVER (ORDER BY month
                                ROWS BETWEEN UNBOUNDED PRECEDING
                                         AND UNBOUNDED FOLLOWING), 2)         AS grand_total,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY month)
             / SUM(revenue) OVER (ORDER BY month
                                  ROWS BETWEEN UNBOUNDED PRECEDING
                                           AND UNBOUNDED FOLLOWING), 1)       AS pct_of_year_to_date
FROM monthly
ORDER BY month
LIMIT 9
""")

**Step by step:**

1. `SUM(revenue) OVER (ORDER BY month)` is the running total. The frame is implied — everything up to this row.
2. `AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` averages this month and the two
   before it. The first two rows average over fewer months, because there is nothing earlier — which is right,
   though it does mean the start of a moving-average line is noisier than the rest.
3. `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` restores the whole-partition total even though
   there is an `ORDER BY`. Without that frame the column would be another running total.
4. The last column divides the running total by the grand total, so it is the cumulative share of the period —
   the shape of an S-curve on a chart.
5. Four different frames over the same window in one query, and one pass over the data. That efficiency is the
   real argument for windows over self-joins.

## 16. NTILE and Percent Rank — Cutting a Population Into Buckets

`NTILE(n)` splits the ordered rows into `n` buckets as evenly as it can and labels each row with its bucket
number. `NTILE(4)` is quartiles, `NTILE(10)` deciles, `NTILE(100)` percentiles.

`PERCENT_RANK()` gives each row its relative position from 0 to 1. `CUME_DIST()` gives the proportion of rows at
or below it.

This is customer segmentation in one line. "Top 25% of customers by spend" is `NTILE(4) OVER (ORDER BY spend
DESC) = 1`, and it needs no thresholds decided in advance and no maintenance when the numbers move.

Two things to know. `NTILE` splits by **row count**, not by value, so if the rows do not divide evenly the
earlier buckets get one extra each. And rows with identical values can land in different buckets — `NTILE` does
not respect ties, so use `RANK` if that matters to you.

In [ ]:
q("""
WITH customer_spend AS (
    SELECT c.customer_id,
           c.name,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS spend
    FROM customers c
    JOIN orders o      ON o.customer_id = c.customer_id
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY c.customer_id, c.name
),
bucketed AS (
    SELECT customer_id,
           name,
           spend,
           NTILE(4)       OVER (ORDER BY spend DESC) AS quartile,
           ROUND(PERCENT_RANK() OVER (ORDER BY spend DESC), 3) AS pct_rank
    FROM customer_spend
)
SELECT quartile,
       COUNT(*)                AS customers,
       ROUND(MIN(spend), 2)    AS min_spend,
       ROUND(MAX(spend), 2)    AS max_spend,
       ROUND(SUM(spend), 2)    AS total_spend,
       ROUND(100.0 * SUM(spend) / SUM(SUM(spend)) OVER (), 1) AS pct_of_revenue
FROM bucketed
GROUP BY quartile
ORDER BY quartile
""")

**Step by step:**

1. `customer_spend` gets one row per customer who has ever ordered, with their total. The nine who never
   ordered are absent — an inner join, deliberately, because you cannot rank someone who has no spend.
2. `NTILE(4) OVER (ORDER BY spend DESC)` labels the biggest spenders 1 and the smallest 4. The bucket sizes come
   out as evenly as the row count allows.
3. `PERCENT_RANK()` gives the same information continuously: 0 for the top customer, 1 for the bottom.
4. The outer query groups by the quartile to show what each is worth. If the top quartile is a large share of
   revenue, that is the Pareto shape almost every customer base has.
5. `SUM(SUM(spend)) OVER ()` looks wrong and is not: the inner `SUM` aggregates within the quartile, the outer
   window sums those four results. A window function applied on top of an aggregate is legal, because windows
   run after grouping.

## 17. Conditional Aggregation — A Cross-Tab Without PIVOT

You met `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` in the basics. Widen it and you have a pivot table: one row per
group, one **column** per category.

```sql
SELECT month,
       SUM(CASE WHEN channel = 'web'   THEN revenue ELSE 0 END) AS web,
       SUM(CASE WHEN channel = 'app'   THEN revenue ELSE 0 END) AS app,
       ...
GROUP BY month
```

Standard SQL has no portable `PIVOT`, and this is what everybody writes instead. It works in every database.

Its one limitation is that the columns are hard-coded — you must know the categories when you write the query.
For four channels that is fine, and it is the normal case: a report has fixed columns. When the categories are
genuinely unknown, you either generate the SQL from a list you queried first, or you return the long format and
pivot in pandas with `df.pivot_table(...)`.

Use `ELSE 0` for sums you want to read as zero, and no `ELSE` (so it falls to `NULL`) for averages, where a zero
would drag the average down instead of being ignored.

In [ ]:
q("""
WITH order_revenue AS (
    SELECT o.order_id,
           o.channel,
           strftime('%Y-%m', o.order_date)                   AS month,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
      AND o.order_date >= '2024-07-01'
    GROUP BY o.order_id, o.channel, month
)
SELECT month,
       ROUND(SUM(CASE WHEN channel = 'web'   THEN revenue ELSE 0 END), 2) AS web,
       ROUND(SUM(CASE WHEN channel = 'app'   THEN revenue ELSE 0 END), 2) AS app,
       ROUND(SUM(CASE WHEN channel = 'store' THEN revenue ELSE 0 END), 2) AS store,
       ROUND(SUM(CASE WHEN channel = 'phone' THEN revenue ELSE 0 END), 2) AS phone,
       ROUND(SUM(revenue), 2)                                             AS total,
       COUNT(DISTINCT CASE WHEN channel = 'store' THEN order_id END)      AS store_orders
FROM order_revenue
GROUP BY month
ORDER BY month
""")

**Step by step:**

1. `order_revenue` fixes the grain first: one row per order, with its channel, month and value. Pivoting a
   fanned-out result would multiply every cell.
2. Each `SUM(CASE WHEN channel = ... )` contributes an order's revenue to exactly one column and zero to the
   others. Four columns, one pass over the data.
3. `SUM(revenue)` with no condition gives the row total, which is the check: the four channel columns must add
   up to it.
4. `COUNT(DISTINCT CASE WHEN channel = 'store' THEN order_id END)` has no `ELSE`, so non-store orders become
   `NULL` and `COUNT` ignores them. That is the idiom for a conditional count of distinct things.
5. To turn this sideways instead — one row per month per channel — drop the `CASE` columns and
   `GROUP BY month, channel`. That long format is what you would hand to a plotting library.

## 18. Cohorts and Retention — Dates in Anger

A **cohort** groups people by when they started, and then measures them over time. It answers the question a
monthly order count cannot: *are the customers we won in March still buying?*

The recipe is always the same three steps:

1. Find each customer's **first** month — their cohort.
2. For every order, compute how many months after that cohort it happened — the **age**.
3. Count distinct customers per cohort per age.

Step 2 is the fiddly one. `strftime('%Y-%m', d)` gives you a label like `'2024-03'`, which you cannot subtract.
Convert both months to a single number instead: `year * 12 + month`. The difference of two of those is the
number of months between them, and it crosses year boundaries without any special handling.

In [ ]:
q("""
WITH first_order AS (
    SELECT customer_id,
           MIN(order_date) AS first_date
    FROM orders
    WHERE status <> 'cancelled'
    GROUP BY customer_id
),
activity AS (
    SELECT o.customer_id,
           strftime('%Y-%m', f.first_date) AS cohort,
           (CAST(strftime('%Y', o.order_date) AS INTEGER) * 12
            + CAST(strftime('%m', o.order_date) AS INTEGER))
         - (CAST(strftime('%Y', f.first_date)  AS INTEGER) * 12
            + CAST(strftime('%m', f.first_date)  AS INTEGER)) AS months_since_first
    FROM orders o
    JOIN first_order f ON f.customer_id = o.customer_id
    WHERE o.status <> 'cancelled'
)
SELECT cohort,
       COUNT(DISTINCT CASE WHEN months_since_first = 0 THEN customer_id END) AS month_0,
       COUNT(DISTINCT CASE WHEN months_since_first BETWEEN 1 AND 3  THEN customer_id END) AS months_1_3,
       COUNT(DISTINCT CASE WHEN months_since_first BETWEEN 4 AND 6  THEN customer_id END) AS months_4_6,
       COUNT(DISTINCT CASE WHEN months_since_first > 6 THEN customer_id END) AS later
FROM activity
WHERE cohort >= '2023-01' AND cohort <= '2023-06'
GROUP BY cohort
ORDER BY cohort
""")

**Step by step:**

1. `first_order` finds each customer's first non-cancelled order date. `MIN` on a `YYYY-MM-DD` text column
   works because that format sorts correctly as text.
2. `activity` joins every order back to its customer's first date. The `cohort` label is the month of that first
   order — the same value on all of a customer's rows.
3. The month arithmetic is the important line. `year * 12 + month` turns `'2024-03'` into 24291; subtract the
   same for the cohort month and you have the age in months. December to January is 1, with nothing special
   written to handle it.
4. `COUNT(DISTINCT CASE WHEN ... THEN customer_id END)` counts customers, not orders, in each age band. Without
   `DISTINCT` a customer who ordered three times in the window would count three times.
5. `month_0` is the cohort's size — everybody is active in the month they first ordered, by definition. Every
   later column read against it is a retention rate, and a cohort table's whole value is that each row is
   comparable to the ones below it.

## 19. Views — Giving a Query a Name

A **view** is a stored query that behaves like a table:

```sql
CREATE VIEW order_totals AS SELECT ... ;
SELECT * FROM order_totals WHERE ...
```

It stores no data. Every time you query the view the underlying query runs, so a view is never stale and never
faster.

Views earn their place when a piece of logic — "revenue excludes cancelled orders and accounts for discount" —
must be identical everywhere. Define it once, and everyone who uses the view gets the same definition. Copies
of that expression scattered across twelve dashboards is how two teams end up reporting different revenue.

A **materialized view** does store its results, and needs refreshing. PostgreSQL and Oracle have them; SQLite
does not, and the usual substitute is a table you rebuild on a schedule — section 21.

Views nest, and deeply nested views are a well-known way to build something nobody can debug. Two levels is
plenty.

In [ ]:
run("""
DROP VIEW IF EXISTS order_totals;

CREATE VIEW order_totals AS
SELECT o.order_id,
       o.customer_id,
       o.order_date,
       o.status,
       o.channel,
       SUM(i.quantity * i.unit_price * (1 - i.discount)) AS order_total,
       COUNT(*)                                          AS lines
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.status <> 'cancelled'
GROUP BY o.order_id, o.customer_id, o.order_date, o.status, o.channel;
""")

print(q("SELECT COUNT(*) AS orders, ROUND(SUM(order_total), 2) AS revenue FROM order_totals")
      .to_string(index=False))

q("""
SELECT channel,
       COUNT(*)                       AS orders,
       ROUND(AVG(order_total), 2)     AS avg_order,
       ROUND(AVG(lines), 2)           AS avg_lines
FROM order_totals
WHERE order_date >= '2024-01-01'
GROUP BY channel
ORDER BY avg_order DESC
""")

**Step by step:**

1. `DROP VIEW IF EXISTS` first, so the cell can be run twice. Views are schema objects and `CREATE` fails if one
   already exists.
2. The view holds the definition of "an order and what it was worth", including the two decisions that are easy
   to get wrong elsewhere: cancelled orders excluded, discount applied.
3. Querying it looks exactly like querying a table. The engine substitutes the definition and runs the combined
   query.
4. The second query filters and groups the view without knowing anything about `order_items`. That is the point
   — the fan-out risk is handled inside the view, and callers cannot reintroduce it.
5. Because a view is just a query, `WHERE order_date >= '2024-01-01'` is pushed down into it rather than the
   whole view being built and then filtered. Databases are good at this; you do not have to help.

## 20. Data Quality Checks You Can Run in SQL

Before you trust a table, interrogate it. These five questions catch most of what goes wrong, and they are all
short:

| Question | Shape |
| --- | --- |
| Are the keys unique? | `GROUP BY key HAVING COUNT(*) > 1` |
| Are there duplicate *entities*? | `GROUP BY email HAVING COUNT(*) > 1` |
| Are there orphans? | `LEFT JOIN parent ... WHERE parent.id IS NULL` |
| How complete is each column? | `COUNT(*) - COUNT(column)` |
| Are the values sane? | `MIN`, `MAX`, and a count of the impossible ones |

A duplicate **key** is a broken constraint. A duplicate **entity** — the same person with two customer ids — is
a business problem the database cannot see, and our data has two of them. Only the second kind needs judgment
about what "the same" means; here it is the email address.

Run these as one stacked query and you have a report card you can re-run whenever the data lands.

In [ ]:
print(q("""
SELECT 'customers'          AS check_name, COUNT(*) AS n FROM customers
UNION ALL SELECT 'customers with no city',    COUNT(*) FROM customers WHERE city IS NULL
UNION ALL SELECT 'duplicate emails',          COUNT(*) FROM (
              SELECT email FROM customers GROUP BY email HAVING COUNT(*) > 1)
UNION ALL SELECT 'orders with no payment',    COUNT(*) FROM orders o
              WHERE NOT EXISTS (SELECT 1 FROM payments p WHERE p.order_id = o.order_id)
UNION ALL SELECT 'items with no order',       COUNT(*) FROM order_items i
              WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.order_id = i.order_id)
UNION ALL SELECT 'products never ordered',    COUNT(*) FROM products p
              WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.product_id = p.product_id)
UNION ALL SELECT 'price differs from list',   COUNT(*) FROM order_items i
              JOIN products p ON p.product_id = i.product_id WHERE i.unit_price <> p.price
""").to_string(index=False))

q("""
SELECT c.email,
       COUNT(*)                        AS accounts,
       GROUP_CONCAT(c.customer_id)     AS customer_ids,
       MIN(c.signup_date)              AS first_signup,
       MAX(c.signup_date)              AS second_signup
FROM customers c
GROUP BY c.email
HAVING COUNT(*) > 1
ORDER BY c.email
""")

**Step by step:**

1. Each `UNION ALL` arm is one check, returning a label and a number. Stacking them gives a single report card
   instead of seven separate results to read.
2. Every arm has the same two columns in the same order. The names come from the first `SELECT`, which is why
   only that one needs the `AS` aliases.
3. `orders with no payment` returns 52 and `items with no order` returns 0. The first is a business fact —
   cancelled and unpaid orders — and the second would be a genuine integrity failure. Knowing which is which is
   the job.
4. `price differs from list` finds order lines charged at something other than today's price. That is correct
   behaviour, not a bug: prices change, and the order records what was actually charged.
5. The last query names the duplicate people. `GROUP_CONCAT` collapses the two customer ids into one string —
   `STRING_AGG` in PostgreSQL, and the equivalent is worth looking up in whatever you use, because a list of
   offending ids is what makes a check actionable.

## 21. Bulk Changes — INSERT SELECT, UPSERT and Transactions

Three things you need once you are writing data rather than just reading it.

**`INSERT ... SELECT`** loads the result of a query straight into a table, without a round trip through your
program. It is how a summary table gets built.

**Upsert** — insert, or update if the row already exists:

```sql
INSERT INTO t (id, x) VALUES (1, 5)
ON CONFLICT (id) DO UPDATE SET x = excluded.x;
```

`excluded` is the row you tried to insert. SQLite and PostgreSQL share this syntax; MySQL spells it
`ON DUPLICATE KEY UPDATE`. It is what makes a load **idempotent** — run it twice and the result is the same,
which matters enormously for anything on a schedule.

**Transactions** group statements so that either all of them happen or none do. `BEGIN`, then `COMMIT` to keep
the work or `ROLLBACK` to throw it away. Anything that changes two tables and would be wrong if only one
succeeded belongs in a transaction.

In [ ]:
run("""
DROP TABLE IF EXISTS customer_summary;

CREATE TABLE customer_summary (
    customer_id INTEGER PRIMARY KEY,
    orders      INTEGER NOT NULL,
    revenue     REAL    NOT NULL,
    last_order  TEXT
);

INSERT INTO customer_summary (customer_id, orders, revenue, last_order)
SELECT c.customer_id,
       COUNT(t.order_id),
       ROUND(COALESCE(SUM(t.order_total), 0), 2),
       MAX(t.order_date)
FROM customers c
LEFT JOIN order_totals t ON t.customer_id = c.customer_id
GROUP BY c.customer_id;
""")
print("loaded:", q("SELECT COUNT(*) AS n FROM customer_summary")["n"][0], "rows")

# re-running the load must not duplicate anything -- that is what the upsert buys you
run("""
INSERT INTO customer_summary (customer_id, orders, revenue, last_order)
SELECT c.customer_id, COUNT(t.order_id), ROUND(COALESCE(SUM(t.order_total), 0), 2), MAX(t.order_date)
FROM customers c
LEFT JOIN order_totals t ON t.customer_id = c.customer_id
GROUP BY c.customer_id
ON CONFLICT (customer_id) DO UPDATE SET
    orders     = excluded.orders,
    revenue    = excluded.revenue,
    last_order = excluded.last_order;
""")
print("after re-running:", q("SELECT COUNT(*) AS n FROM customer_summary")["n"][0], "rows")

before = q("SELECT ROUND(SUM(revenue), 2) AS total FROM customer_summary")["total"][0]
con.execute("UPDATE customer_summary SET revenue = 0")        # opens a transaction
during = q("SELECT ROUND(SUM(revenue), 2) AS total FROM customer_summary")["total"][0]
con.rollback()                                                 # ... and throws it away
after = q("SELECT ROUND(SUM(revenue), 2) AS total FROM customer_summary")["total"][0]
print(f"before {before:,.2f} | inside the transaction {during:,.2f} | after rollback {after:,.2f}")

q("SELECT * FROM customer_summary ORDER BY revenue DESC LIMIT 5")

**Step by step:**

1. `INSERT ... SELECT` fills the summary table from a query over the `order_totals` view built in section 19.
   The `LEFT JOIN` keeps the nine customers who never ordered, and `COALESCE(SUM(...), 0)` gives them a real
   zero rather than `NULL`.
2. Running the same load again would normally fail on the primary key. `ON CONFLICT (customer_id) DO UPDATE`
   turns it into an update instead, so the row count stays at 60 — the load is idempotent.
3. `excluded.orders` refers to the value the failed insert was carrying. Without it you would have to repeat
   the whole expression in the `SET` clause.
4. The transaction demo uses `con.execute` rather than `run`, because `run` uses `executescript`, which commits
   as it goes. The `UPDATE` zeroes every row, the query inside sees zeros, and `rollback()` restores the lot.
5. Nothing was lost because nothing was committed. That is the guarantee: until `COMMIT`, your changes are
   yours alone and can be abandoned completely.

## 22. If You Move to PostgreSQL or MySQL

The heavy machinery of this level — CTEs, window functions, `EXISTS`, set operators — is standard SQL and works
the same everywhere. What moves is the vocabulary around the edges.

| Task | SQLite (here) | PostgreSQL | MySQL 8+ |
| --- | --- | --- | --- |
| Month bucket | `strftime('%Y-%m', d)` | `to_char(d, 'YYYY-MM')` | `DATE_FORMAT(d, '%Y-%m')` |
| Truncate to month | `date(d, 'start of month')` | `DATE_TRUNC('month', d)` | `DATE_FORMAT(d, '%Y-%m-01')` |
| Months between | `y * 12 + m` arithmetic | `AGE(a, b)` or the same arithmetic | `TIMESTAMPDIFF(MONTH, a, b)` |
| Collect into a list | `GROUP_CONCAT(x)` | `STRING_AGG(x, ',')` | `GROUP_CONCAT(x)` |
| Upsert | `ON CONFLICT ... DO UPDATE` | `ON CONFLICT ... DO UPDATE` | `ON DUPLICATE KEY UPDATE` |
| Set difference | `EXCEPT` | `EXCEPT` | `EXCEPT` (8.0.31+) |
| Materialized view | not available | `CREATE MATERIALIZED VIEW` | not available |
| `FULL OUTER JOIN` | 3.39+ | yes | no — emulate with `UNION` |
| Filtered aggregate | `SUM(CASE WHEN ...)` | `COUNT(*) FILTER (WHERE ...)` | `SUM(CASE WHEN ...)` |

Two behavioural notes that matter at this level:

- **`GROUP BY` strictness.** PostgreSQL rejects any selected column that is neither grouped nor aggregated.
  SQLite picks an arbitrary row. Write for PostgreSQL and both are happy.
- **Window function support.** SQLite has had it since 3.25 and MySQL since 8.0. On MySQL 5.7 none of sections
  12 to 17 exist, and you are back to correlated subqueries and self-joins — which is exactly why they are
  still worth knowing.

In [ ]:
print("SQLite:", q("SELECT sqlite_version() AS v")["v"][0])

q("""
SELECT channel,
       COUNT(*)                                                     AS orders,
       SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END)         AS returned,
       -- PostgreSQL:  COUNT(*) FILTER (WHERE status = 'returned')
       ROUND(AVG(COUNT(*)) OVER (), 1)                              AS avg_orders_per_channel
FROM orders
GROUP BY channel
ORDER BY orders DESC
""")

**Step by step:**

1. Window functions need SQLite 3.25 or later. If a query in this notebook fails to parse, check this number
   first — very old Python builds ship an older engine.
2. `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` is the portable conditional count. The commented `FILTER` clause is
   PostgreSQL's cleaner spelling of the same thing, and SQLite supports it too from 3.30.
3. `AVG(COUNT(*)) OVER ()` is a window function over an aggregate — the average of the four per-channel counts.
   Legal because windows run after grouping, and identical in every database that has windows.
4. The `--` comment sits inside the SQL and travels with the query. Leaving the target dialect's spelling in a
   comment is a small kindness to whoever ports it.
5. Everything else in this query is plain standard SQL and would run unchanged on any of the three.

## 23. Writing Queries People Can Read — And This Level's Pitfalls

A query that works is half the job. Six habits that make the other half:

1. **One CTE per step, named for what it produces.** `order_totals`, `per_customer`, `ranked`. Not `t1`, `t2`.
2. **State the grain in a comment** at the top of each CTE: `-- one row per order`. Most join bugs are grain
   bugs, and writing the grain down is when you notice.
3. **Alias every table, prefix every column.** `o.order_date`, never a bare `order_date` in a three-table query.
4. **Filter as early as you can.** A `WHERE` inside the first CTE removes rows before the expensive work.
5. **Put the `ORDER BY` you need for reproducibility**, including a tiebreaker, on anything you will compare.
6. **Build it one CTE at a time**, running each on its own. Do not write forty lines and then debug them.

And the mistakes this level makes possible:

| Pitfall | What happens | Fix |
| --- | --- | --- |
| Joining before aggregating | totals multiplied by the number of child rows | aggregate to one row per key first |
| `NOT IN` over a nullable column | zero rows, silently | `NOT EXISTS` |
| `COUNT(*)` after a `LEFT JOIN` | non-matching rows counted as 1 | `COUNT(right.key)` |
| Filtering a window function in the same `WHERE` | syntax error | compute in a CTE, filter outside |
| `ORDER BY` inside `OVER` on a `SUM` you wanted whole | silent running total | write the frame out |
| `DISTINCT` to fix duplicate rows | hides a fan-out instead of fixing it | find out why the rows doubled |
| Condition on the right table in `WHERE` after a `LEFT JOIN` | the join becomes inner | move it into the `ON` |
| Percentages with integer division | zeroes everywhere | `100.0 *`, not `100 *` |

In [ ]:
# the LEFT JOIN that quietly became an INNER JOIN
in_the_where = q("""
SELECT COUNT(*) AS n
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'delivered'
""")["n"][0]

in_the_on = q("""
SELECT COUNT(*) AS n
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'delivered'
""")["n"][0]

print("condition in WHERE:", in_the_where, "rows -- customers with no delivered order have vanished")
print("condition in ON   :", in_the_on, "rows -- everybody is still here")

# COUNT(*) vs COUNT(column) after a LEFT JOIN
q("""
SELECT COUNT(*)          AS count_star,
       COUNT(o.order_id) AS count_orders,
       COUNT(DISTINCT c.customer_id) AS customers
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
""")

**Step by step:**

1. The first query looks like a `LEFT JOIN` and behaves like an inner one. `WHERE o.status = 'delivered'` is
   tested after the join has filled the unmatched rows with `NULL`, and `NULL = 'delivered'` is not true.
2. Moving the same condition into the `ON` keeps every customer: the join simply does not find a match for
   some, and their columns stay `NULL`. Which one you want depends on the question — the bug is not noticing
   there is a choice.
3. `COUNT(*)` is 309 — the 300 orders plus one placeholder row for each of the nine customers with none.
4. `COUNT(o.order_id)` is 300, because it skips those `NULL`s. That is the number you almost always meant.
5. `COUNT(DISTINCT c.customer_id)` is 60. Three counts, three different correct answers, over exactly the same
   rows — which is why "how many?" is never a complete question on its own.

## Cheat Sheet

| Task | SQL |
| --- | --- |
| Keep unmatched left rows | `FROM a LEFT JOIN b ON b.a_id = a.id` |
| Every combination | `FROM a CROSS JOIN b` |
| Walk a hierarchy | `FROM employees e LEFT JOIN employees m ON m.employee_id = e.manager_id` |
| Fix fan-out | aggregate to one row per key in a CTE, then join |
| Count without fan-out | `COUNT(DISTINCT o.order_id)` |
| Rows with no match | `WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.a_id = a.id)` |
| Rows with a match | `WHERE EXISTS (SELECT 1 FROM b WHERE b.a_id = a.id)` |
| Never use over a nullable column | `NOT IN (SELECT nullable FROM t)` |
| Compare to a whole-table value | `WHERE price > (SELECT AVG(price) FROM products)` |
| Aggregate an aggregate | `FROM (SELECT ... GROUP BY order_id) t GROUP BY channel` |
| Value from the row's own group | `(SELECT AVG(x) FROM t2 WHERE t2.g = t.g)` |
| Name the steps | `WITH step1 AS (...), step2 AS (...) SELECT ...` |
| Stack results | `SELECT ... UNION ALL SELECT ...` |
| In both / in one only | `INTERSECT` / `EXCEPT` |
| Group value on every row | `AVG(price) OVER (PARTITION BY category_id)` |
| Whole-table value on every row | `AVG(price) OVER ()` |
| Number the rows | `ROW_NUMBER() OVER (PARTITION BY g ORDER BY x DESC)` |
| Ties share a number | `RANK()` (gaps) / `DENSE_RANK()` (no gaps) |
| Top N per group | rank in a CTE, then `WHERE rn <= N` |
| Previous / next row | `LAG(x) OVER (ORDER BY d)` / `LEAD(x) OVER (ORDER BY d)` |
| Running total | `SUM(x) OVER (ORDER BY d)` |
| Moving average | `AVG(x) OVER (ORDER BY d ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` |
| Partition total despite `ORDER BY` | `... ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` |
| Quartiles | `NTILE(4) OVER (ORDER BY spend DESC)` |
| Cross-tab column | `SUM(CASE WHEN channel = 'web' THEN revenue ELSE 0 END) AS web` |
| Conditional distinct count | `COUNT(DISTINCT CASE WHEN c THEN id END)` |
| Months between two dates | `(y1 * 12 + m1) - (y2 * 12 + m2)` |
| Name a query | `CREATE VIEW v AS SELECT ...` |
| Load from a query | `INSERT INTO t (...) SELECT ...` |
| Insert or update | `INSERT ... ON CONFLICT (id) DO UPDATE SET x = excluded.x` |
| All or nothing | `BEGIN; ...; COMMIT;` or `ROLLBACK;` |
| Duplicate entities | `GROUP BY email HAVING COUNT(*) > 1` |

## Suggested Learning Path

1. Run section 1, then read sections 2 and 3 together. Fan-out is the most valuable thing in this notebook.
2. Do exercises 2 and 3 before going further. If your revenue number is wrong, everything downstream is.
3. Sections 4 and 5 are anti-joins and the `NOT IN` trap. Type the broken query out yourself and watch it
   return zero.
4. Sections 6 to 9 are subqueries in each position they can occupy. Notice which ones a CTE would read better as.
5. Read section 11 and then go back and rewrite your answers to exercises 7 and 8 as CTEs.
6. Section 12 is the conceptual jump: aggregate without collapsing. Re-read it until `OVER (PARTITION BY ...)`
   feels like `GROUP BY` that kept your rows.
7. Sections 13 to 16 are the window function family. Memorise the top-N-per-group shape; you will write it
   constantly.
8. Section 17 turns rows into columns; section 18 puts it together into a cohort table. These two are the
   report-building sections.
9. Sections 19 to 21 are the write side — views, loads, upserts, transactions.
10. Finish on section 23 and check you can explain all eight pitfalls without looking.
11. Do the exercises in order. The two mini projects at the end are the level in miniature.

## Where to Go Next

- **`sql-advanced`** takes the same queries and asks why they are slow: query plans, indexes, recursive CTEs,
  transactions in earnest, star schemas, and the rewrites that turn a two-minute report into a two-second one.
- Take a report somebody at work already relies on and rewrite it with CTEs and window functions. Check the
  numbers match the old version before you replace it — and when they do not, find out which one was wrong.
- Read the window function chapters of [PostgreSQL's manual](https://www.postgresql.org/docs/current/tutorial-window.html).
  It is the clearest treatment of frames anywhere, and it applies almost word for word to SQLite.

## Practice Next

Open `sql-intermediate-exercises.ipynb`. Every exercise names the guide section it practises, and the checks
tell you when the answer is right. The two mini projects — a cohort retention table and a best-seller report —
are the ones to keep in your portfolio.